In [ ]:
import json
from pathlib import Path

import src

In [ ]:
def load_config(config_path: str | Path = "CONFIG_FILE.json") -> dict:
    """Load the notebook configuration JSON."""
    config_path = Path(config_path)
    with config_path.open("r", encoding="utf-8") as f:
        cfg = json.load(f)

    print(f"Loaded path configuration from {config_path}.")
    return cfg


def build_crawlers(cfg: dict, ignore_hidden: bool = True) -> dict[str, src.mre_metadata_crawler.MREMetadataCrawler]:
    """Build one crawler per scanner setup defined in the config."""
    anchor = Path.cwd()
    root = src.utils.find_root_with_marker(anchor, ".drive_root.txt")

    setups = {
        "3T": {
            "dicom_root": src.utils.combine_paths(root, cfg["RELATIVE_DICOM_ROOT_PATH_3T"]),
            "output_root": src.utils.combine_paths(root, cfg["RELATIVE_OUTPUT_DIR_ROOT_3T"], check_if_exists=False),
            "output_summary_tsv": cfg["METADATA_SUMMARY_TSV_NAME"].format(scanner="3T"),
            "folder_template": cfg["GENERAL_FOLDER_TEMPLATE_3T"],
        },
        "7T": {
            "dicom_root": src.utils.combine_paths(root, cfg["RELATIVE_DICOM_ROOT_PATH_7T"]),
            "output_root": src.utils.combine_paths(root, cfg["RELATIVE_OUTPUT_DIR_ROOT_7T"], check_if_exists=False),
            "output_summary_tsv": cfg["METADATA_SUMMARY_TSV_NAME"].format(scanner="7T"),
            "folder_template": cfg["GENERAL_FOLDER_TEMPLATE_7T"],
        },
    }

    crawlers = {}
    for label, info in setups.items():
        paths = src.mre_metadata_crawler.CrawlPaths(
            label=label,
            dicom_root=Path(info["dicom_root"]),
            output_tsv_path=Path(src.utils.combine_paths(info["output_root"], info["output_summary_tsv"], check_if_exists=False)),
            folder_pattern_template=info["folder_template"].format(dicom_root=info["dicom_root"], subject="{subject}"),
        )
        crawlers[label] = src.mre_metadata_crawler.MREMetadataCrawler(paths, ignore_hidden=ignore_hidden)

    return crawlers


In [ ]:
cfg = load_config("CONFIG_FILE.json")
print()
print("Building crawlers...")
crawlers = build_crawlers(cfg)
for crawler in crawlers.values():
    crawler.print_setup()
    print()

# Run modes:
# - default: only process subjects missing from the TSV
# - subject_override=[...] : process only those subjects
# - force_all_subjects=True : re-scan the complete DICOM tree


In [ ]:
# Set force_all_subjects=True to force a complete re-run for this scanner.
summary3T, detailed3T, updated_summary3T = crawlers["3T"].update_summary(
    verbose=False,
    dry_run=True,
    force_all_subjects=False,
)
# updated_summary3T


In [ ]:
#subject_override_7T = ["phantom_custom_fmre"]  # set to None to use missing-subject mode

summary7T, detailed7T, updated_summary7T = crawlers["7T"].update_summary(
    verbose=True,
    dry_run=False,
    #subject_override=subject_override_7T,
    force_all_subjects=False,
)
updated_summary7T
